# Notebook 01 — Ingestion API → MongoDB et simulation des paiements
## Étape 5 — Connexion à MongoDB

In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=3000)

# Vérifier que le serveur répond
print(client.admin.command("ping"))
print("Version MongoDB :", client.server_info()["version"])

# Sélectionner la base commerce
db = client["commerce"]
print("Base sélectionnée :", db.name)
print("Collections existantes :", db.list_collection_names())

{'ok': 1.0}
Version MongoDB : 7.0.48
Base sélectionnée : commerce
Collections existantes : []


## Atelier 1 (1/2) — Ingestion des ressources API dans MongoDB
Source : API DummyJSON (données de démonstration), endpoint `https://dummyjson.com/<ressource>?limit=0`.
Chaque ressource est insérée dans la collection `raw_<ressource>` de la base `commerce`.

In [2]:
import requests
import pandas as pd
from datetime import datetime

BASE_URL = "https://dummyjson.com"
RESSOURCES = ["products", "carts", "users", "posts", "comments", "recipes", "quotes", "todos"]

date_collecte = datetime.now()
print("Date de collecte :", date_collecte.strftime("%d/%m/%Y %H:%M:%S"))

recap = []

for ressource in RESSOURCES:
    # 1. Récupérer la ressource complète depuis l'API (limit=0 = tous les éléments)
    url = f"{BASE_URL}/{ressource}?limit=0"
    reponse = requests.get(url, timeout=30)
    reponse.raise_for_status()          # erreur explicite si l'API ne répond pas 200
    data = reponse.json()

    documents = data[ressource]         # la liste est sous la clé du même nom
    total_api = data["total"]           # nombre total annoncé par l'API

    # 2. Vider la collection puis insérer (le notebook peut être relancé sans doublons)
    collection = db[f"raw_{ressource}"]
    collection.delete_many({})
    collection.insert_many(documents)

    # 3. Contrôler le volume inséré
    nb_mongo = collection.count_documents({})
    recap.append({
        "ressource": ressource,
        "collection": collection.name,
        "total_api": total_api,
        "documents_recus": len(documents),
        "documents_mongo": nb_mongo,
        "ok": total_api == len(documents) == nb_mongo,
    })

df_recap = pd.DataFrame(recap)
display(df_recap)
print("Total documents insérés :", df_recap["documents_mongo"].sum())
print("Collections dans commerce :", sorted(db.list_collection_names()))

Date de collecte : 26/11/2026 11:28:20


,ressource,collection,total_api,documents_recus,documents_mongo,ok
0,products,raw_products,194,194,194,True
1,carts,raw_carts,208,208,208,True
2,users,raw_users,208,208,208,True
3,posts,raw_posts,251,251,251,True
4,comments,raw_comments,340,340,340,True
5,recipes,raw_recipes,50,50,50,True
6,quotes,raw_quotes,1454,1454,1454,True
7,todos,raw_todos,254,254,254,True


Total documents insérés : 2959
Collections dans commerce : ['raw_carts', 'raw_comments', 'raw_posts', 'raw_products', 'raw_quotes', 'raw_recipes', 'raw_todos', 'raw_users']


## Atelier 1 (2/2) — 1. Import des commandes dans `raw_commandes`
Source : `data/processed/commandes_ecommerce.csv` (800 lignes, 16 colonnes, une ligne = un produit dans un panier).
Le fichier est importé tel quel (collection RAW) : aucune ligne n'est supprimée ni modifiée.

In [3]:
import math
from pathlib import Path

# Racine du projet (le notebook est dans notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV_COMMANDES = ROOT / "data" / "processed" / "commandes_ecommerce.csv"

df_commandes = pd.read_csv(CSV_COMMANDES)
df_commandes["order_date"] = pd.to_datetime(df_commandes["order_date"])   # stockée en vraie date dans Mongo
print("Fichier :", CSV_COMMANDES.name, "-", df_commandes.shape[0], "lignes,", df_commandes.shape[1], "colonnes")

# Conversion en documents ; les valeurs manquantes (NaN) deviennent null dans MongoDB
documents = df_commandes.to_dict("records")
for doc in documents:
    for champ, valeur in doc.items():
        if isinstance(valeur, float) and math.isnan(valeur):
            doc[champ] = None

col_commandes = db["raw_commandes"]
col_commandes.delete_many({})
col_commandes.insert_many(documents)

# ---- Contrôles d'intégrité et de volume ----
somme_mongo = list(col_commandes.aggregate([
    {"$group": {"_id": None, "total": {"$sum": "$discounted_line_total"}}}
]))[0]["total"]

controles = pd.DataFrame([
    {"contrôle": "Nombre de documents",          "CSV": len(df_commandes),                         "MongoDB": col_commandes.count_documents({})},
    {"contrôle": "Paniers distincts (cart_id)",  "CSV": df_commandes["cart_id"].nunique(),         "MongoDB": len(col_commandes.distinct("cart_id"))},
    {"contrôle": "Somme discounted_line_total",  "CSV": round(df_commandes["discounted_line_total"].sum(), 2), "MongoDB": round(somme_mongo, 2)},
    {"contrôle": "Lignes sans marque (brand)",   "CSV": int(df_commandes["brand"].isna().sum()),   "MongoDB": col_commandes.count_documents({"brand": None})},
])
controles["ok"] = controles["CSV"] == controles["MongoDB"]
display(controles)
print("Exemple de document :", col_commandes.find_one({}, {"_id": 0}))

Fichier : commandes_ecommerce.csv - 800 lignes, 16 colonnes


,contrôle,CSV,MongoDB,ok
0,Nombre de documents,800.00,800.00,True
1,Paniers distincts (cart_id),208.00,208.00,True
2,Somme discounted_line_total,3456709.58,3456709.58,True
3,Lignes sans marque (brand),377.00,377.00,True


Exemple de document : {'order_id': 'CART-00001', 'cart_id': 1, 'user_id': 1, 'username': 'emilys', 'product_id': 162, 'product_title': 'Blue Frock', 'category': 'tops', 'brand': None, 'quantity': 4, 'unit_price': 29.99, 'line_total': 119.96, 'discount_percentage': 12.13, 'discounted_line_total': 105.41, 'cart_total': 13037.88, 'cart_discounted_total': 11510.81, 'order_date': datetime.datetime(2025, 3, 2, 0, 0)}


## Atelier 1 (2/2) — 2. Simulation des paiements (`raw_payments`)
**Processus :** les commandes sont regroupées par `cart_id` (agrégation MongoDB) pour obtenir **un seul enregistrement par panier**, puis une tentative de paiement unique est générée pour chacun.

**Répartition probabiliste :** tirage aléatoire pondéré (`random.choices`) avec PAID 70 %, REFUSED 20 %, PENDING 10 %. Une **graine fixe (42)** rend la simulation reproductible : même résultat à chaque exécution et sur chaque poste.

**Éléments repris du panier :** `cart_id` et `order_id` (commande), `user_id` (client), `cart_discounted_total` (montant payé après remise, identique sur toutes les lignes d'un panier donc pris une seule fois).

**Date de paiement :** la date la plus récente des lignes du panier. ⚠️ `order_date` est synthétique : cette date n'a pas de valeur métier réelle.

In [4]:
import random

col_payments = db["raw_payments"]

# 1. Un enregistrement par panier, calculé par MongoDB à partir de raw_commandes
paniers = list(col_commandes.aggregate([
    {"$group": {
        "_id": "$cart_id",
        "order_id":     {"$first": "$order_id"},
        "user_id":      {"$first": "$user_id"},
        "montant":      {"$first": "$cart_discounted_total"},
        "montant_min":  {"$min":   "$cart_discounted_total"},   # contrôle : le total doit être
        "montant_max":  {"$max":   "$cart_discounted_total"},   # identique sur toutes les lignes
        "nb_lignes":    {"$sum": 1},
        "nb_articles":  {"$sum": "$quantity"},
        "date_paiement": {"$max": "$order_date"},
    }},
    {"$sort": {"_id": 1}},   # ordre fixe → tirage reproductible
]))
assert all(p["montant_min"] == p["montant_max"] for p in paniers), "Total de panier incohérent"
print("Paniers distincts :", len(paniers))

# 2. Tirage pondéré reproductible
STATUTS = ["PAID", "REFUSED", "PENDING"]
POIDS   = [70, 20, 10]
rng = random.Random(42)

paiements = []
for i, p in enumerate(paniers, start=1):
    statut = rng.choices(STATUTS, weights=POIDS)[0]
    paiements.append({
        "payment_id":     f"PAY-{i:05d}",
        "cart_id":        p["_id"],
        "order_id":       p["order_id"],
        "user_id":        p["user_id"],
        "amount":         round(p["montant"], 2),
        "status":         statut,
        "amount_collected": round(p["montant"], 2) if statut == "PAID" else 0.0,
        "payment_date":   p["date_paiement"],
        "nb_lignes":      p["nb_lignes"],
        "nb_articles":    p["nb_articles"],
    })

# 3. Insertion avec un index unique sur cart_id (empêche physiquement un 2e paiement par panier)
col_payments.drop()
col_payments.create_index("cart_id", unique=True)
col_payments.insert_many(paiements)
print("Paiements insérés :", col_payments.count_documents({}))

Paniers distincts : 208
Paiements insérés : 208


In [5]:
ids_commandes = set(col_commandes.distinct("cart_id"))
ids_paiements = set(col_payments.distinct("cart_id"))
doublons = list(col_payments.aggregate([
    {"$group": {"_id": "$cart_id", "n": {"$sum": 1}}},
    {"$match": {"n": {"$gt": 1}}},
]))
total_paiements = sum(p["amount"] for p in col_payments.find({}, {"amount": 1}))

controles_pay = pd.DataFrame([
    {"contrôle": "1 paiement par panier (208)",            "valeur": col_payments.count_documents({}),            "ok": col_payments.count_documents({}) == len(ids_commandes)},
    {"contrôle": "Aucun cart_id en double",                 "valeur": len(doublons),                                "ok": len(doublons) == 0},
    {"contrôle": "Tous les paniers ont un paiement",       "valeur": len(ids_commandes - ids_paiements),           "ok": ids_commandes == ids_paiements},
    {"contrôle": "Statuts autorisés uniquement",            "valeur": str(sorted(col_payments.distinct("status"))), "ok": set(col_payments.distinct("status")) <= set(STATUTS)},
    {"contrôle": "Σ montants = Σ discounted_line_total",    "valeur": round(total_paiements, 2),                    "ok": round(total_paiements, 2) == round(somme_mongo, 2)},
])
display(controles_pay)

,contrôle,valeur,ok
0,1 paiement par panier (208),208,True
1,Aucun cart_id en double,0,True
2,Tous les paniers ont un paiement,0,True
3,Statuts autorisés uniquement,"['PAID', 'PENDING', 'REFUSED']",True
4,Σ montants = Σ discounted_line_total,3456709.58,True


In [6]:
# Nombre et montant par statut, calculés par MongoDB
analyse = list(col_payments.aggregate([
    {"$group": {
        "_id": "$status",
        "nb_paiements":   {"$sum": 1},
        "montant_total":  {"$sum": "$amount"},
        "montant_encaisse": {"$sum": "$amount_collected"},
    }},
    {"$sort": {"nb_paiements": -1}},
]))

df_analyse = pd.DataFrame(analyse).rename(columns={"_id": "statut"})
df_analyse["% paiements"] = (df_analyse["nb_paiements"] / df_analyse["nb_paiements"].sum() * 100).round(1)
df_analyse["% montant"]   = (df_analyse["montant_total"] / df_analyse["montant_total"].sum() * 100).round(1)
df_analyse["% théorique"] = df_analyse["statut"].map({"PAID": 70, "REFUSED": 20, "PENDING": 10})
df_analyse[["montant_total", "montant_encaisse"]] = df_analyse[["montant_total", "montant_encaisse"]].round(2)
display(df_analyse)

ca_encaisse = df_analyse["montant_encaisse"].sum()
ca_potentiel = df_analyse["montant_total"].sum()
print(f"Montant total des paniers    : {ca_potentiel:,.2f}")
print(f"Chiffre d'affaires encaissé  : {ca_encaisse:,.2f}  ({ca_encaisse / ca_potentiel * 100:.1f} % du total)")
print(f"Montant non encaissé         : {ca_potentiel - ca_encaisse:,.2f}")

,statut,nb_paiements,montant_total,montant_encaisse,% paiements,% montant,% théorique
0,PAID,150,2669235.61,2669235.61,72.1,77.2,70
1,REFUSED,37,423555.75,0.00,17.8,12.3,20
2,PENDING,21,363918.22,0.00,10.1,10.5,10


Montant total des paniers    : 3,456,709.58
Chiffre d'affaires encaissé  : 2,669,235.61  (77.2 % du total)
Montant non encaissé         : 787,473.97


### Analyse des paiements simulés

| Statut | Paiements | Part | Montant associé |
|---|---:|---:|---:|
| PAID | 150 | 72,1 % | 2 669 235,61 |
| REFUSED | 37 | 17,8 % | 423 555,75 |
| PENDING | 21 | 10,1 % | 363 918,22 |
| **Total** | **208** | 100 % | **3 456 709,58** |

- **Chiffre d'affaires encaissé : 2 669 235,61**, soit environ 77 % du montant total des paniers. Seuls les paiements PAID sont encaissés ; REFUSED et PENDING n'apportent aucun chiffre d'affaires dans ce scénario.
- **Écart à la théorie :** les proportions observées (72 / 18 / 10 %) sont proches des probabilités cibles (70 / 20 / 10 %). L'écart vient du tirage aléatoire sur un petit échantillon de 208 paniers ; il diminuerait avec plus de paniers.
- **La part du montant encaissé (77 %) est supérieure à la part des paiements PAID (72 %)** : les paniers acceptés sont en moyenne plus chers que les autres. C'est un effet du hasard (le tirage ne dépend pas du montant), amplifié par quelques paniers très chers (catégorie vehicle).
- **Absence de double comptage :** un index unique sur `cart_id` empêche tout second paiement pour un même panier, et la somme des 208 paiements est égale à la somme des 800 lignes de commandes (3 456 709,58).